In [193]:
import sys
sys.path.append('/data/bionets/og86asub/alternet-project/alternet/')
import pandas as pd
import alternet.data_preprocessing as preprocessing
from alternet.annotation import map_tf_ids
import alternet.annotation as annotation
import alternet.postprocessing as postprocessing

from alternet.inference import *
import yaml
import time
from typing import Any, Dict
import numpy as np

import seaborn as sns

def write_dict_to_yaml(data: Dict[str, Any], filepath: str):
    """Writes a Python dictionary to a YAML file."""

    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)




In [ ]:
config_file = "/data/bionets/og86asub/alternet-project/alternet/configs/MAGNet_NF.yaml"

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)


biomart_path = '/data/bionets/og86asub/alternet-project/alternet/data/biomart.txt'
tf_list_path = '/data/bionets/og86asub/alternet-project/alternet/data/allTFs_hg38.txt'
appris_df = pd.read_csv(config['appris'], sep='\t')

biomart = pd.read_csv(biomart_path, sep='\t')
tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
tf_list = map_tf_ids(tf_list, biomart)


transcript_data = pd.read_csv(config['transcript_data'], index_col=0)

# Subset to protein coding isoforms
protein_coding_isoforms = list(appris_df[appris_df['Transcript type'] == 'protein_coding']['Transcript ID'])
transcript_data = transcript_data[transcript_data.transcript_id.isin(protein_coding_isoforms)]



gene_data = transcript_data.groupby('gene_id').sum()
gene_data = gene_data.drop(columns={'transcript_id'})

gene_data.index.name = 'gene_id'
gene_data = gene_data.reset_index()

sample_attributes = pd.read_csv(config['sample_attributes'])
sample_attributes = sample_attributes.loc[:, ['sample_name', 'etiology']]

conditions = ['DCM']

runs = 10
## Subset the samples of interest
for condi in conditions:

    samples = sample_attributes[sample_attributes['etiology'] == config['tissue']]
    samples = samples['sample_name'].tolist()

    gene_data_cp = gene_data.copy(deep=True)
    transcript_data_cp = transcript_data.copy(deep=True)
    ## Get unified gene and transcript table
    gene_data_cp = gene_data_cp.loc[:, ['gene_id'] + samples ]
    transcript_data_cp = transcript_data_cp.loc[:,['gene_id', 'transcript_id'] + samples]


    gene_target_names = list(tf_list['Gene stable ID'].unique())


    gene_data_cp = gene_data_cp.set_index('gene_id')
    gene_data_cp = gene_data_cp.T

    # scale data!!
    gene_data_cp_scaled = standardize_dataframe(gene_data_cp)

    transcript_data_cp = transcript_data_cp.set_index('transcript_id')
    transcript_data_cp = transcript_data_cp.drop('gene_id', axis=1)
    transcript_data_cp = transcript_data_cp.T

    # scale data!!
    transcript_data_cp_scaled = standardize_dataframe(transcript_data_cp)


    #start = time.monotonic()
    #canonical_grn = inference(gene_data=gene_data_cp, tf_list=gene_target_names, target_names = 'all', n_runs=runs)
    #end = time.monotonic()
    #runtime = {'canonical': end-start}
    
    #canonical_grn.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_canonical.tsv", header = True)

    #hybrid_data = create_hybrid_data(transcript_data_cp, gene_data_cp, tf_list)
    #hybrid_tf_names = list(tf_list['Transcript stable ID'].unique())
    #target_names = list(gene_data_cp.columns)

    #start = time.monotonic()
    #as_aware_grn = inference(gene_data=hybrid_data, tf_list=hybrid_tf_names, target_names=target_names, n_runs = runs)
    #runtime['as_aware'] = time.monotonic()-start

    #as_aware_grn.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_as_aware.tsv", header = True) 
    #write_dict_to_yaml(runtime, f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_runtime.yaml")

    tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
    tf_list = map_tf_ids(tf_list, biomart)

    #filter aggregate
    as_aware_grn = pd.read_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_as_aware.tsv", index_col=0)
    canonical_grn = pd.read_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_canonical.tsv", index_col=0)

    transcript_mapper = annotation.create_transcript_mapping(biomart)
    as_aware_grn = postprocessing.map_transcript_to_gene(as_aware_grn, transcript_mapper)

    as_aware_grn = postprocessing.create_edge_key(as_aware_grn)
    canonical_grn = postprocessing.create_edge_key(canonical_grn, source_column='source', target_column='target')
    canonical_grn = canonical_grn.rename(columns={'source': 'source_gene'})


    # Categorize edges into gene-unique, isoform-unique, common
    common_edges = postprocessing.get_common_edges(canonical_grn, as_aware_grn)
    gene_unique, isoform_unique = postprocessing.get_diff(canonical_grn, as_aware_grn)

    filtering_tracker = {'as_aware_base': as_aware_grn.shape[0], 'canonical_base': canonical_grn.shape[0],  'common_base': common_edges.shape[0], 'gene_unique_base': gene_unique.shape[0], 'isoform_unique_base': isoform_unique.shape[0]}

    #common_edges, abs_threshold_c, f_common = postprocessing.filter_aggregated(common_edges, threshold_importance=0.2, threshold_frequency=10)
    gene_unique, abs_threshold_g, f_gene = postprocessing.filter_aggregated(gene_unique, threshold_importance=0.2, threshold_frequency=10)
    isoform_unique, abs_threshold_i, f_isoform = postprocessing.filter_aggregated(isoform_unique, threshold_importance=0.2, threshold_frequency=10)


    filtering_tracker['common_filter_1'] = f_common
    filtering_tracker['gene_unique_filter_1'] = f_gene
    filtering_tracker['isoform_unique_filter_1'] = f_isoform


    isoform_categories = postprocessing.isoform_categorization(transcript_data_cp, gene_data_cp, tf_list)
    isoform_categories_sub = isoform_categories.loc[:, ['Gene stable ID', 'Transcript stable ID', 'percentage', 'isoform_category']]

    isoform_unique, iso_plausibility_filter  = postprocessing.plausibility_filtering(isoform_unique, isoform_categories_sub)
    filtering_tracker['isoform_unique_plausibility_filter'] = iso_plausibility_filter
    tf_database = annotation.create_transcipt_annotation_database(tf_list=tf_list, appris_path= config['appris'], digger_path=config['digger'])


    isoform_unique = annotation.merge_annotations_to_grn(isoform_unique, tf_database, transcript_column='source_transcript')
    isoform_unique  = isoform_unique.drop(columns={'Gene stable ID', 'Transcript stable ID', 'Transcript type'})
    isoform_unique.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_unique_isofoms.tsv")

    gene_categories = postprocessing.get_gene_cases(isoform_categories_sub)
    gene_unique, filtering_info_gene_plaus = postprocessing.plausibility_filtering_gene_unique(gene_unique, gene_categories)
    filtering_tracker['gene_unique_plausibility_filter'] = filtering_info_gene_plaus
    gene_unique.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_unique_genes.tsv")

    write_dict_to_yaml(filtering_tracker, f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_filtering_tracker.yaml")

    merged_edges = get_common_edges(common_edges)
    consistent, ambigous = split_by_isoform_category(merged_edges, gene_categories)
    consistent = plausibility_filtering_common_edges_dominant(consistent)

    consistent.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_consistent_both.tsv")

    likely_isoform_specific, ambigous = find_likely_isoform_specific(ambigous)
    likely_isoform_specific.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_likely_isoform_specific.tsv")

    
    likely_gene_specific, ambigous = find_likely_gene_specific(ambigous)
    likely_gene_specific.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_likely_gene_specific.tsv")

    ambigous.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_ambigous.tsv")

    gene_to_transcript_mapping = annotation.create_filtered_gene_to_transcripts_mapping(biomart, gene_list = gene_data_cp_scaled.columns, transcript_list = transcript_data_cp_scaled.columns)
    correlation_collector = compute_isoform_gene_correlations(transcript_data_cp_scaled, gene_data_cp_scaled, gene_to_transcript_mapping)

        






In [6]:
del canonical_grn
del as_aware_grn

In [ ]:
def get_common_edges(common_edges):
    common_edges_ge = common_edges[pd.isna(common_edges.source_transcript)].copy()
    common_edges_t = common_edges[~pd.isna(common_edges.source_transcript)].copy()
    merged_edges = common_edges_t.merge(common_edges_ge, on='edge_key', suffixes=['_te', '_ge'])
    merged_edges = merged_edges.rename(columns={'source_transcript_te': 'source_transcript', 'source_gene_te': 'source_gene', 'target_te': 'target_gene'})
    merged_edges = merged_edges.loc[:, ['source_transcript', 'source_gene', 'target_gene', 'edge_key', 'frequency_te', 'mean_importance_te', 'median_importance_te',  'frequency_ge', 'mean_importance_ge', 'median_importance_ge']]
    return merged_edges


def plausibility_filtering_common_edges_dominant(consistent, threshold_upper=1.5, threshold_lower=0.5, threshold_frequency = 10):
    """
    Plausbility filtering for edges which are found in both networks
    As the isoforms are either dominant or there is a single isoform it is required
        - the are at the same high frequency in both networks
        - fold change of importances is about 1.0

    """
    consistent['fc'] = consistent['mean_importance_te']/consistent['mean_importance_ge']
    consistent['mean_importance_te_ge'] = consistent[['median_importance_te', 'median_importance_ge']].mean(axis=1)
    consistent = consistent[(consistent['frequency_te']>=threshold_frequency )& (consistent['frequency_ge']>=threshold_frequency)]
    consistent = consistent[(consistent['fc']<threshold_upper) & (consistent['fc']>threshold_lower)].sort_values('mean_importance_te', ascending=False)
    consistent = consistent.reset_index()
    consistent = consistent.drop(columns = ['index'])
    consistent = consistent.sort_values('mean_importance_te_ge',ascending=False)

    return consistent.copy()

def split_by_isoform_category(merged_edges, gene_categories):
    merged_edges = merged_edges.merge(gene_categories, left_on='source_gene', right_on='Gene stable ID')
    merged_edges = merged_edges.loc[:, ['source_transcript', 'source_gene', 'target_gene', 'edge_key', 'frequency_te', 'mean_importance_te', 'median_importance_te',  'frequency_ge', 'mean_importance_ge', 'median_importance_ge', 'gene_category']]

    consistent = merged_edges[merged_edges.gene_category.isin(['single', 'dominant'])].copy()
    ambigous = merged_edges[merged_edges.gene_category.isin(['balanced', 'non-dominant'])].copy()
    return consistent, ambigous


def find_likely_isoform_specific(ambigous, lf_threshold = 2, frequency_threshold = 10):
    ambigous['fc']  = ambigous['mean_importance_te']/ambigous['mean_importance_ge']
    ambigous['absolute_difference'] = np.abs(ambigous['mean_importance_te']- ambigous['mean_importance_ge'])
    likely_isoform_specific = ambigous[(ambigous['fc']>lf_threshold)  & (ambigous['frequency_ge']>=frequency_threshold)].copy()
    remove_keys = likely_isoform_specific.edge_key
    likely_isoform_specific = likely_isoform_specific.sort_values('median_importance_te', ascending=False)
    likely_isoform_specific = likely_isoform_specific.reset_index()
    likely_isoform_specific = likely_isoform_specific.drop(columns = ['index'])

    ambigous = ambigous[~ambigous.edge_key.isin(remove_keys)].copy()

    return likely_isoform_specific, ambigous

def find_likely_gene_specific(ambigous, frequency_threshold = 10, lf_threshold = 0.5):
    ambigous['fc']  = ambigous['mean_importance_te']/ambigous['mean_importance_ge']
    ambigous['gene_specific'] = ambigous.groupby('edge_key')['fc'].transform('max') < lf_threshold
    likely_gene_specific = ambigous[ambigous.gene_specific]
    remove_keys_edge  = likely_gene_specific.edge_key

    likely_gene_specific = likely_gene_specific[(likely_gene_specific['frequency_ge']>=frequency_threshold)].copy()
    likely_gene_specific.sort_values('median_importance_ge')
    likely_gene_specific = likely_gene_specific.reset_index()
    likely_gene_specific = likely_gene_specific.drop(columns = ['index'])

    ambigous = ambigous[~ambigous.edge_key.isin(remove_keys_edge)].copy()
    
    return likely_gene_specific, ambigous
    

KeyboardInterrupt: 

In [121]:
for i in range(200):
    print(likely_isoform_specific.target_x[i])

ENSG00000149541
ENSG00000177034
ENSG00000132549
ENSG00000007520
ENSG00000166710
ENSG00000130811
ENSG00000011132
ENSG00000067208
ENSG00000116871
ENSG00000074071
ENSG00000130309
ENSG00000011009
ENSG00000133275
ENSG00000129968
ENSG00000157637
ENSG00000105298
ENSG00000126461
ENSG00000176619
ENSG00000066777
ENSG00000214655
ENSG00000167967
ENSG00000130479
ENSG00000137692
ENSG00000023228
ENSG00000133065
ENSG00000102898
ENSG00000167548
ENSG00000079313
ENSG00000074071
ENSG00000167397
ENSG00000109906
ENSG00000198752
ENSG00000161016
ENSG00000139343
ENSG00000108175
ENSG00000196576
ENSG00000115760
ENSG00000106400
ENSG00000197136
ENSG00000198865
ENSG00000272333
ENSG00000173039
ENSG00000175662
ENSG00000100425
ENSG00000272333
ENSG00000073417
ENSG00000198865
ENSG00000129968
ENSG00000153443
ENSG00000265241
ENSG00000161960
ENSG00000047410
ENSG00000023287
ENSG00000108557
ENSG00000123144
ENSG00000167657
ENSG00000164880
ENSG00000130702
ENSG00000008710
ENSG00000198816
ENSG00000198793
ENSG00000179632
ENSG0000